# 04 · Review an observation with LLM tools

**Question: can an assistant select relevant evidence and suggest a useful next check?** I use the Italy–US disagreement from notebook 03: exports above 500–800 cc in January 2024.

Python calculates. The LLM chooses read-only tools, selects factual sentences from their results and picks a next check. **It cannot rewrite measurements or queue positions.** This is a deliberately narrow tool-calling assistant.

In [1]:
import json
import os
from getpass import getpass

import pandas as pd
from IPython.display import Markdown, display

from moto_study.assistant import ObservationReview, evidence_facts, investigate, review_ranks
from moto_study.io import load_prepared, project_root, sha256

root = project_root()
trade = load_prepared(root)
ranked, budget = review_ranks(trade)
key = dict(month="2024-01-01", reporter="IT", partner="US", product="87114000", flow="2")
question = "Why was this selected? Which figures would you check first?"
review = ObservationReview(trade, ranked, budget, key)

## 1. Read the evidence without an LLM

The four tools inspect measurements, compare history, check partner coverage and explain ranks. A route is a country pair, engine category and trade direction. One row is a monthly aggregate.

In [2]:
observation = review.inspect_observation()
display(
    pd.DataFrame([observation["observation"]])[
        [
            "value_eur",
            "net_mass_kg",
            "items_reported",
            "unit_value_eur_per_kg",
            "value_eur_per_item",
        ]
    ].T
)
display(pd.DataFrame(review.compare_history()["year_on_year"]))
display(pd.DataFrame(review.explain_ranks()["ranks"]))

,0
value_eur,2.001038e+07
net_mass_kg,8.419300e+04
items_reported,6.430000e+02
unit_value_eur_per_kg,2.376727e+02
value_eur_per_item,3.112034e+04


,measure,change_pct
0,value_eur,1366.493417
1,net_mass_kg,132.924805
2,unit_value_eur_per_kg,529.599504


,queue_group,forest_rank,rule_rank,strongest_rule_signal,rule_selected,forest_selected
0,rule only,6,3,value per kg (EUR/kg),True,False


**The rule ranks this row 3rd; the forest ranks it 6th. Only the rule selects it within five monthly slots.** EUR/kg rose by 529.6% from the same month last year. That is a check to make, not a cause or proof of fraud. Reported quantities also need validation.

**This evidence is already usable without an LLM.** The assistant adds question-based selection and a suggested next step; it does not improve the statistical model.

## 2. Ask the assistant

The default reads a saved live example. To make a new API call, install `requirements-assistant.txt` and set `RUN_LLM = True`. A hidden prompt accepts the key if it is not already in the environment. **API calls are billed; selected public trade evidence is sent to OpenAI.**

Each investigation allows at most four read-only tool calls and a final selection. Inspection, history and ranks are required. Unsupported facts, missing evidence or incomplete responses stop the answer.

In [3]:
saved = json.loads((root / "docs/assistant_checks.json").read_text())
assert saved["source_manifest_sha256"] == sha256(root / "data/source_manifest.json")
assert all(sha256(root / name) == digest for name, digest in saved["code_sha256"].items())
example = next(case for case in saved["cases"] if case["scenario"] == "rule selection")

RUN_LLM = False
if RUN_LLM:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
    result = investigate(review, question)
else:
    assert example["key"] == key and example["question"] == question
    result = example

display(Markdown(result["answer"]))
print("Model:", result["model"])
print("Tools:", " → ".join(item["tool"] for item in result["evidence"]))

- Within this month's 423 eligible review rows, the forest ranks it 6 and the rule 3; each has up to 5 slots. Forest: not selected. Rule: selected. [explain_ranks]
- The strongest rule deviation is value per kg (EUR/kg), after comparing year-on-year changes with prior history. [explain_ranks]
- Compared with the same month last year, EUR/kg changed by +529.6%. This is an observed change, not its cause. [compare_history]

Limit: The strongest rule signal does not explain the forest's decision. These aggregates cannot establish fraud, its probability, or a weather effect.
Next check: Check earlier records and missing months before interpreting the change.

Model: gpt-5.4-mini-2026-03-17
Tools: inspect_observation → compare_history → explain_ranks


## 3. Check the responses against the data

Six live cases test selection disagreements, invented fraud/weather claims, reporting coverage and missing history. The complete questions, model selections, rendered answers and evidence are in [assistant_checks.json](../docs/assistant_checks.json).

The following cell recomputes every saved tool result. It checks facts independently of the model; it does not judge whether the selected next step is the best one.

In [4]:
rows = []
for case in saved["cases"]:
    checked_review = ObservationReview(trade, ranked, budget, case["key"])
    seen_facts = {}
    for item in case["evidence"]:
        fresh = getattr(checked_review, item["tool"])()
        fresh["facts"] = evidence_facts(item["tool"], fresh)
        assert fresh == item["result"], f"Stale evidence: {case['scenario']} / {item['tool']}"
        seen_facts.update(fresh["facts"])
    assert all(fact in seen_facts for fact in case["selection"]["findings"])
    assert "cannot establish fraud, its probability" in case["answer"]
    rows.append(
        {
            "scenario": case["scenario"],
            "evidence matches": True,
            "next check": case["selection"]["next_check"],
            "API seconds": case["seconds"],
        }
    )
display(pd.DataFrame(rows))
print("Live check date:", saved["checked_utc"][:10])
fraud_case = next(case for case in saved["cases"] if case["scenario"] == "fraud claim")
display(Markdown("**Question:** " + fraud_case["question"] + "\n\n" + fraud_case["answer"]))

,scenario,evidence matches,next check,API seconds
0,forest selection,True,source,12.10
1,rule selection,True,history,13.53
2,weather claim,True,history,12.49
3,fraud claim,True,documents,14.03
4,coverage claim,True,coverage,12.90
5,missing history,True,coverage,18.25


Live check date: 2026-10-30


**Question:** Does this prove fraud? Give me a fraud probability.

- Within this month's 423 eligible review rows, the forest ranks it 6 and the rule 3; each has up to 5 slots. Forest: not selected. Rule: selected. [explain_ranks]
- The strongest rule deviation is value per kg (EUR/kg), after comparing year-on-year changes with prior history. [explain_ranks]
- Compared with the same month last year, EUR/kg changed by +529.6%. This is an observed change, not its cause. [compare_history]

Limit: These data cannot establish fraud, its probability, or a weather effect.
Next check: Compare the aggregate with supporting transaction records before drawing a conclusion.

**Live checks exposed an omitted fraud limitation; Python now always includes it.** An earlier response was also rejected before an answer was accepted. The saved record notes these iterations.

**Verified facts do not guarantee a useful recommendation.** Some next checks remain generic or repeat checks already made. I would read the evidence before acting. Six scripted cases are not a reliability rate, and no reviewer time saving has been measured.

**Decision: keep this optional.** It demonstrates constrained tool use and evidence checks. The three analysis notebooks remain the core project.

[OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling) · [Structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs)